# Churn Prediction Inteligente

Primera etapa del proyecto: limpieza, analisis inicial y feature engineering usando PySpark.

## Importacion de librerias

In [ ]:
import sys
from pathlib import Path

from pyspark.sql import functions as F

# Permite importar funciones desde la carpeta src del proyecto.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src" / "data_preprocessing.py").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("No se encontro la carpeta src del proyecto")
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))

from data_preprocessing import (
    analizar_dataset,
    cargar_dataset,
    crear_sesion_spark,
    crear_variables_derivadas,
    guardar_dataset,
    limpiar_dataset,
)

spark = crear_sesion_spark("ChurnPredictionNotebook")
spark.sparkContext.setLogLevel("ERROR")

## Carga del dataset

In [ ]:
ruta_dataset = PROJECT_ROOT / "data" / "raw" / "dataset_original.csv"
df = cargar_dataset(spark, ruta_dataset)

df.show(5, truncate=False)

## Analisis inicial

In [ ]:
analisis = analizar_dataset(df)

print(f"Cantidad de filas y columnas: {analisis['filas_columnas']}")
print("\nNombre de las columnas:")
print(analisis["columnas"])
print("\nTipos de datos:")
print(analisis["tipos_datos"])
print("\nValores nulos por columna:")
print(analisis["valores_nulos"])
print(f"\nRegistros duplicados: {analisis['duplicados']}")
print(f"\nValores unicos de Churn: {analisis['valores_unicos_churn']}")
print("\nDistribucion de Churn:")
print(analisis["distribucion_churn"])

print("\nEsquema del dataset:")
df.printSchema()

print("\nResumen estadistico:")
df.describe().show()

## Limpieza de datos

In [ ]:
# Convierte TotalCharges a numerico, imputa vacios con la mediana,
# elimina duplicados y transforma Churn a 0/1 usando PySpark.
df_limpio = limpiar_dataset(df)

df_limpio.select("customerID", "TotalCharges", "Churn").show(5, truncate=False)
print("Valores unicos de Churn despues de limpiar:")
df_limpio.select("Churn").distinct().show()
print("Duplicados despues de limpiar:", df_limpio.count() - df_limpio.dropDuplicates().count())

## Feature engineering

In [ ]:
# Crea las variables derivadas solicitadas para enriquecer el dataset.
df_limpio = crear_variables_derivadas(df_limpio)

df_limpio.select(
    "customerID",
    "NumServicios",
    "ClienteNuevo",
    "CargoPromedioPorMes",
    "ContratoMensual",
).show(5, truncate=False)

## Revision final del dataset

In [ ]:
print("Filas y columnas finales:", (df_limpio.count(), len(df_limpio.columns)))
print("\nColumnas finales:")
print(df_limpio.columns)

print("\nValores nulos finales:")
valores_nulos_finales = {
    columna: df_limpio.filter(F.col(columna).isNull()).count()
    for columna in df_limpio.columns
}
print(valores_nulos_finales)

print("\nDistribucion final de Churn:")
df_limpio.groupBy("Churn").count().show()

## Exportacion del dataset limpio

In [ ]:
ruta_salida = PROJECT_ROOT / "processed" / "telco_churn_limpio.csv"
guardar_dataset(df_limpio, ruta_salida)

print(f"Dataset limpio guardado en: {ruta_salida}")
df_limpio.select(
    "NumServicios",
    "ClienteNuevo",
    "CargoPromedioPorMes",
    "ContratoMensual",
).show(5, truncate=False)